# M5 · Backfill `fact_trip`

**Goal:** load the full history of `staging.path` into `warehouse.fact_trip` in 30-day chunks.

**SQL file:** `sql/10_fact_trip_incremental.sql` (parameters: `:window_start`, `:window_end`, `:etl_run_id`).

**Exit criterion:**
- `warehouse.etl_watermark.last_event_time` for `fact_trip` ≥ `MAX(staging.path.begin_path_time)`.
- Row count in the fact table looks right (orders of magnitude vs staging).
- Validation V1 (row count) passes.

> **Run order:** this notebook assumes the facts before it (lower number) have already been loaded.


In [ ]:
# === Setup ===
from __future__ import annotations
import sys, pathlib
# Make the src/ package importable even if pip install -e was skipped.
PROJECT_ROOT = pathlib.Path().resolve().parents[1] if pathlib.Path().resolve().name != 'accent-fleet-analytics' else pathlib.Path().resolve()
for candidate in (PROJECT_ROOT, PROJECT_ROOT.parent):
    src = candidate / 'src'
    if src.exists() and str(src) not in sys.path:
        sys.path.insert(0, str(src))
        break

from accent_fleet.config import settings
from accent_fleet.db import get_engine, transaction
from sqlalchemy import text

s = settings()
print(f"DB = {s.pg_user}@{s.pg_host}:{s.pg_port}/{s.pg_database}")
print(f"Schemas: staging={s.pg_schema_staging}, warehouse={s.pg_schema_warehouse}, marts={s.pg_schema_marts}")


## 2. Inputs — config, SQL, and the backfill plan

In [ ]:
from accent_fleet.config import load_pipeline_config
from accent_fleet.db.sql_loader import load_sql
from datetime import datetime, timedelta

cfg = load_pipeline_config()
CHUNK_DAYS = cfg['window']['backfill_chunk_days']
MIN_TIME   = datetime.fromisoformat(cfg['window']['min_event_time'].replace('Z','+00:00')).replace(tzinfo=None)
print('chunk_days =', CHUNK_DAYS, '  min_time =', MIN_TIME)

with get_engine().connect() as conn:
    end_time = conn.execute(text("SELECT MAX(begin_path_time) FROM staging.path")).scalar_one()
print('staging max event-time =', end_time)

print('\n----- SQL file preview -----')
print(load_sql('10_fact_trip_incremental.sql')[:600], '...')


## 3. Execute — loop 30-day chunks, call `load_fact_incremental` for each

In [ ]:
import time, pandas as pd
from accent_fleet.transforms.facts import load_fact_incremental
from accent_fleet.pipeline.run_log import begin_run, end_run

run_id = begin_run(mode='notebook:01_load_fact_trip')
print('etl_run_id =', run_id)

progress = []
cursor = MIN_TIME
t0 = time.time()
try:
    while cursor < end_time:
        next_cursor = min(cursor + timedelta(days=CHUNK_DAYS), end_time)
        res = load_fact_incremental('fact_trip', run_id=run_id, window_end=next_cursor)
        progress.append({
            'window_start': cursor, 'window_end': next_cursor,
            'rows_loaded': res.rows_loaded, 'new_max': res.new_max_event_time,
        })
        cursor = next_cursor
    end_run(run_id, status='success', rows_loaded=sum(p['rows_loaded'] for p in progress))
except Exception as exc:
    end_run(run_id, status='failed', error_message=str(exc))
    raise
print(f'done in {time.time()-t0:.1f}s — {len(progress)} chunks')
pd.DataFrame(progress).tail(10)


## 4. Inspect — watermark, counts, quarantine

In [ ]:
import pandas as pd
with get_engine().connect() as conn:
    wm = pd.read_sql(text("SELECT * FROM warehouse.etl_watermark WHERE table_name='fact_trip'"), conn)
    fc = conn.execute(text('SELECT COUNT(*) FROM warehouse.fact_trip')).scalar_one()
    sc = conn.execute(text('SELECT COUNT(*) FROM staging.path')).scalar_one()
    q  = pd.read_sql(text("""SELECT rule_id, COUNT(*) AS n
                               FROM warehouse.quarantine_rejected
                               WHERE source_table='path' AND etl_run_id = :rid
                               GROUP BY rule_id"""), conn, params={'rid': run_id})
print(f'staging.path: {sc:,} rows')
print(f'warehouse.fact_trip: {fc:,} rows')
display(wm); display(q)
assert wm['last_event_time'].iloc[0] is not None, 'watermark did not advance'
print('OK — backfill of fact_trip complete.')
